# Document shadow removal: training and evaluation on Colab

Runtime > Change runtime type > **T4 GPU**, then run the cells in order. Keep this tab open.

Everything is saved to Google Drive (`MyDrive/shadow_project/outputs`). If Colab disconnects, run all cells again: finished steps are skipped and training resumes from the last epoch.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Get the data: first tries the official SD7K OneDrive link directly; if that fails, uses a copy of the zip on your Google Drive (`ZIP`).

In [ ]:
ZIP = '/content/drive/MyDrive/SD7K.zip'   # only used if the official link fails
import os, base64, subprocess, glob
share = 'https://1drv.ms/u/s!Avp0JjwC1wv5jhx6TxHfjlL4BMuA?e=GFTcze'  # official SD7K link
direct = 'https://api.onedrive.com/v1.0/shares/u!' + base64.b64encode(share.encode()).decode().rstrip('=').replace('/', '_').replace('+', '-') + '/root/content'
if not os.path.exists('/content/data') and not os.path.exists('/content/sd7k.zip'):
    print('Trying the official link ...')
    r = subprocess.run(['wget', '-q', '--show-progress', '-O', '/content/sd7k.zip', direct])
    ok = r.returncode == 0 and os.path.getsize('/content/sd7k.zip') > 1e9
    if not ok:
        os.remove('/content/sd7k.zip') if os.path.exists('/content/sd7k.zip') else None
        if os.path.exists(ZIP):
            print('Official link failed; copying the zip from your Drive ...'); subprocess.run(['cp', ZIP, '/content/sd7k.zip'], check=True)
        else:
            print('Official link failed and ZIP not found. Upload the SD7K zip to Drive and set ZIP. Zips found:', glob.glob('/content/drive/MyDrive/**/*.zip', recursive=True)[:20])
print('zip ready' if os.path.exists('/content/sd7k.zip') or os.path.exists('/content/data') else 'no data yet')

In [ ]:
%%bash
set -e
if [ ! -d /content/data ]; then
  mkdir -p /content/data
  echo 'Unzipping ...'; unzip -q /content/sd7k.zip -d /content/data && rm /content/sd7k.zip
fi
find /content/data -maxdepth 3 -type d | head
df -h /content | tail -1

In [ ]:
!rm -rf /content/code && git clone -q https://github.com/mbengue0/document-shadow-removal /content/code && ls /content/code

Full run: dataset list, pair checks, crop cache, 3 trainings, classical baseline, evaluation and figures.

In [ ]:
!cd /content/code && python code.py --data /content/data --out /content/drive/MyDrive/shadow_project/outputs --cache /content/cache --workers 2

In [ ]:
from IPython.display import Image, display, Markdown
out = '/content/drive/MyDrive/shadow_project/outputs'
display(Markdown(open(f'{out}/summary_sd7k.md').read()))
for f in ['fig_training_curves.png', 'fig_metrics_sd7k.png', 'fig_examples_sd7k.png']:
    display(Image(f'{out}/{f}'))